In [1]:
import os, pickle
from pathlib import Path
import pandas as pd

from sklearn.datasets import load_iris

## Imputation module
import xgboost
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

## Main module
from synthcity.plugins import Plugins
from synthcity.utils.serialization import save_to_file, load_from_file

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.getcwd()

/mnt/hum01-home01/p88346bn/.conda/envs/synthcity/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[KeOps] Warning : Cuda libraries were not detected on the system or could not be loaded ; using cpu only mode


'/mnt/hum01-home01/p88346bn/test/project/bayes-ctgan-fix/benchmarks'

### Data Prep

In [2]:
dfs = [] 
data_dir = '/mnt/hum01-home01/p88346bn/test/project/tab-ddpm/'
country_names = ['Canada','Fiji','UK','Rwanda','Indonesia','Adult','Churn','Insurance','Credit']
for i in range(9):
    dfs.append(pd.read_csv(data_dir+'census-data2/'+country_names[i]+'.csv',dtype=str))
    print(country_names[i])

Canada
Fiji
UK
Rwanda
Indonesia
Adult
Churn
Insurance
Credit


In [3]:
cont_cols_canada = ['AGE','HRSWK','INCTOT','WKSWORK']
cat_cols_canada = [i for i in dfs[0].columns if i not in cont_cols_canada]
dfs[0][cont_cols_canada] = dfs[0][cont_cols_canada].astype(float)

iimp = IterativeImputer(
    estimator = xgboost.XGBRegressor(),
    random_state = 42,
    verbose = 2,
)

dfs[0][cont_cols_canada] = iimp.fit_transform(dfs[0][cont_cols_canada]) ## imputation to remove NA

cont_col_adult = ['age','fnlwgt','capital-gain','capital-loss','hours-per-week']
cont_col_churn = ['CreditScore', 'Age', 'Balance','EstimatedSalary']
cont_col_insurance = ['age','bmi','charges'] ## y = regression
cont_col_credit = ['months_loan_duration','amount','age']

## adult

cont_col_noncensus = [cont_col_adult, cont_col_churn, cont_col_insurance, cont_col_credit]
discrete_col_noncensus = [[j for j in dfs[i].columns if j not in cont_col_noncensus[i-5]] for i in range(5,9)]

discrete_columns = [cat_cols_canada,dfs[1].columns.tolist(),dfs[2].columns.tolist(),
                    dfs[3].columns.tolist(),dfs[4].columns.tolist()] + discrete_col_noncensus
cont_columns = [cont_cols_canada,['AGE'],['AGE'],None, None] + cont_col_noncensus

for i in range(5,9):
    dfs[i][cont_columns[i]] = dfs[i][cont_columns[i]].astype(float)
    
y_cols = ['TENURE','TENURE','TENURE','MARST','MARST',
          'income','Exited','charges','checking_balance']

[IterativeImputer] Completing matrix with shape (32149, 4)
[IterativeImputer] Ending imputation round 1/10, elapsed time 3.87
[IterativeImputer] Change: 36205.83135963276, scaled tolerance: 537.8 
[IterativeImputer] Ending imputation round 2/10, elapsed time 7.77
[IterativeImputer] Change: 368.32977199554443, scaled tolerance: 537.8 
[IterativeImputer] Early stopping criterion reached.


### Running model

In [ ]:
## their default n_iter is 1000
# Details are in 
# https://synthcity.readthedocs.io/en/latest/generators.html#general-purpose
# batch_size = 500
# lr = 2e-3
# our lr in gan = 2e-4


# plugin = Plugins().get("bayesian_network")
# plugin = Plugins().get("privbayes", epsilon = 1.0)

### run on gpu
# plugin = Plugins().get("tvae", n_iter = 10, device='cuda',
#                       workspace = 'babi/run')
# plugin = Plugins().get("rtvae", n_iter = 100, device='cuda')
# plugin = Plugins().get("nflow", n_iter = 100, device='cuda')
# plugin = Plugins().get("ddpm", n_iter=100, is_classification=True)
# plugin.fit(dfs[4])

In [ ]:
# # for model in ["tvae", "ddpm", "nflow", "rtvae"]:
# for model in ["ddpm"]:
#     for idx in range(1):
#         plugin = Plugins().get(model, n_iter = 200, device=device,
#                                batch_size = 500)
#         print(f"Running {model} on {country_names[idx]} dataset using {plugin.device}.")
#         plugin.fit(dfs[idx])

#         save_path = Path(f"results/{country_names[idx]}/{model}/model.pkl")
#         save_path.parent.mkdir(parents=True, exist_ok=True)
        
#         save_to_file(save_path, plugin)
#         print("Model Saved to:", save_path)
        
# #         for i in range(5):
# #             # generate again without fit ulang
# #             data_syn = plugin.generate(count=len(dfs[idx])).dataframe()
# #             data_syn.to_csv(f"results/{country_names[idx]}/{model}/gen_data_{i}.csv", index=False)

In [ ]:
# for model in ["tvae", "ddpm", "nflow", "rtvae"]:
# param_dict = {'ctgan': {'generator_n_layers_hidden': 3, 
#                         'generator_n_units_hidden': 640,
#                         'discriminator_n_layers_hidden': 3, 
#                         'discriminator_n_units_hidden': 640},
#              'tvae': {'encoder_n_layers_hidden': 3, 
#                       'encoder_n_units_hidden': 768,
#                       'decoder_n_layers_hidden': 3, 
#                       'decoder_n_units_hidden': 768
#                      },
#              'nflow': {'n_layers_hidden': 2,
#                        'n_units_hidden': 1280}
#              }

param_dict = {'ctgan': {'generator_n_layers_hidden': 2, 
                        'generator_n_units_hidden': 256,
                        'discriminator_n_layers_hidden': 2, 
                        'discriminator_n_units_hidden': 256},
             'nflow': {'n_layers_hidden': 3,
                       'n_units_hidden': 768,
                       'batch_norm': False}
             }

pl1 = []
for model in ["nflow"]:
# for model in ["ctgan", "tvae", "nflow"]:
    for idx in range(1):
        plugin = Plugins().get(model, n_iter = 1, device=device,
                               batch_size = 4096, **param_dict[model])
        print(f"Running {model} on {country_names[idx]} dataset using {plugin.device}.")
        plugin.fit(dfs[idx])
        print(f"num of {model} parameter: {sum(p.numel() for p in plugin.model.parameters())}")
#         pl1.append(plugin)
        

#         save_path = Path(f"results/{country_names[idx]}/{model}/model.pkl")
#         save_path.parent.mkdir(parents=True, exist_ok=True)
        
#         save_to_file(save_path, plugin)
#         print("Model Saved to:", save_path)
        
#         for i in range(5):
#             # generate again without fit ulang
#             data_syn = plugin.generate(count=len(dfs[idx])).dataframe()
#             data_syn.to_csv(f"results/{country_names[idx]}/{model}/gen_data_{i}.csv", index=False)

[2026-06-03T12:13:46.074799+0100][1800369][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-06-03T12:13:46.077316+0100][1800369][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-06-03T12:13:46.079331+0100][1800369][CRITICAL] module plugin_great load failed
[2026-06-03T12:13:46.081677+0100][1800369][CRITICAL] module disabled: /mnt/hum01-home01/p88346bn/.conda/envs/synthcity/lib/python3.9/site-packages/synthcity/plugins/generic/plugin_goggle.py


Running nflow on Canada dataset using cpu.


100%|██████████| 1/1 [00:48<00:00, 48.50s/it]

num of nflow parameter: 13013271


In [22]:
sum(p.numel() for p in plugin.model.parameters())

35511006

In [20]:
plugin.model

TabularGAN(
  (model): GAN(
    (generator): MLP(
      (model): Sequential(
        (0): SkipConnection(LinearLayer)(
          (model): Sequential(
            (0): Linear(in_features=2063, out_features=2048, bias=True)
            (1): ReLU()
          )
        )
        (1): SkipConnection(LinearLayer)(
          (model): Sequential(
            (0): Linear(in_features=4111, out_features=4111, bias=True)
            (1): ReLU()
          )
        )
        (2): SkipConnection(LinearLayer)(
          (model): Sequential(
            (0): Linear(in_features=8222, out_features=8222, bias=True)
            (1): ReLU()
          )
        )
        (3): SkipConnection(LinearLayer)(
          (model): Sequential(
            (0): Linear(in_features=16444, out_features=16444, bias=True)
            (1): ReLU()
          )
        )
        (4): Linear(in_features=32888, out_features=70, bias=True)
        (5): MultiActivationHead()
      )
      (loss): MSELoss()
    )
    (discriminato

In [ ]:
### gan

param_dict = {'GAN': 'generator_n_layers_hidden': 4, 
                     'generator_n_units_hidden': 1024,
                     'discriminator_n_layers_hidden' = 4, 
                     'discriminator_n_units_hidden' = 1024}

In [11]:
for i in range(5):
    # generate again without fit ulang
    data_syn = plugin.generate(count=len(dfs[idx])).dataframe()
    data_syn.to_csv(f"results/{country_names[idx]}/{model}/gen_data_{i}.csv", index=False)


KeyboardInterrupt



In [21]:
from typing import Any, Optional, Tuple

def generate_tabddpm(count: int, sample_batch: int = 40000, cond: Any = None) -> pd.DataFrame:
    plugin.model.diffusion.eval()
    if cond is not None:
        cond = torch.tensor(cond, dtype=torch.long, device=self.device)
    sample = plugin.model.diffusion.sample_all(count, cond, max_batch_size = sample_batch).detach().cpu().numpy()
    df = pd.DataFrame(sample, columns=plugin.model.feature_names_out)
    df = df[plugin.model.feature_names]
    df = plugin.encoder.inverse_transform(df)
    return df 

In [45]:
from synthcity.plugins.core.schema import Schema


def generate2(plugin, count):
    # We use the training schema for the generation
    gen_constraints = plugin.training_schema().as_constraints()
    
    syn_schema = Schema.from_constraints(gen_constraints)

    def generate_tabddpm(count: int) -> pd.DataFrame:
        plugin.model.diffusion.eval()
        sample_batch = 40000

        sample = plugin.model.diffusion.sample_all(count, None, max_batch_size = sample_batch).detach().cpu().numpy()
        df = pd.DataFrame(sample, columns=plugin.model.feature_names_out)
        df = df[plugin.model.feature_names]
        df = plugin.encoder.inverse_transform(df)
        return df
    
    X_syn = plugin._safe_generate(generate_tabddpm, count, syn_schema)
    
#     X_syn = plugin._generate(count=count, syn_schema=syn_schema, **kwargs)

    if X_syn.is_tabular():
        if plugin.compress_dataset:
            X_syn = X_syn.decompress(plugin.compress_context)
        if plugin._data_encoders is not None:
            X_syn = X_syn.decode(plugin._data_encoders)

    # The dataset is decompressed here, we can use the public schema
    gen_constraints = plugin.schema().as_constraints()
    
    if not X_syn.satisfies(gen_constraints) and self.strict:
        raise RuntimeError(
            f"Plugin {self.name()} failed to meet the synthetic constraints."
        )

    if plugin.strict:
        X_syn = X_syn.match(gen_constraints)

    return X_syn

In [35]:
from typing import Any, Optional, Tuple
aa = generate2(1000)

In [36]:
aa

,AGE,HRSWK,INCTOT,WKSWORK,ABIDENT,SEX,TENURE,URBAN,WKFULL,BPLMOM,...,MARST,TRANWORK,RELATE,DEGREE,OCCISCO,YRIMM,MINORITY,RELIG,IND,BPL
0,44,40.000000,21542.310514,52.000000,2,2,1,2,1,1,...,3,4,6,2,5,NaN,13,16,NaN,7
1,NaN,0.000000,-33300.000000,52.000000,2,1,1,1,1,3,...,1,NaN,6,2,NaN,NaN,NaN,1,1,1
2,22,40.000000,70926.154015,52.000000,2,1,1,1,1,1,...,2,1,1,2,2,NaN,13,6,19,7
3,NaN,40.000000,65288.609078,52.000000,1,1,1,1,1,1,...,2,1,2,7,2,NaN,13,16,19,7
4,26,40.000000,31838.965380,52.000000,2,1,2,2,1,3,...,1,3,6,9,4,7,1,11,16,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,45,40.000000,34478.835236,52.000000,2,2,1,1,1,1,...,1,1,3,3,3,NaN,13,4,19,7
996,NaN,17.859472,12811.843750,44.531971,2,2,1,2,NaN,1,...,1,NaN,3,NaN,NaN,NaN,13,16,NaN,7
997,33,28.000000,53487.984202,52.000000,2,1,1,2,1,2,...,2,1,2,8,5,NaN,4,NaN,6,7
998,18,17.859472,12811.843750,0.000000,2,2,1,2,NaN,1,...,1,1,1,2,5,NaN,13,2,7,7


In [14]:
for i in range(1):
    # generate again without fit ulang
    print(f'sampling seed {i}')
    data_syn = generate_tabddpm(plugin, count=len(dfs[idx]), sample_batch = min(len(dfs[idx]), 40000))
    data_syn.to_csv(f"results/{country_names[idx]}/{model}/gen_data_{i}.csv", index=False)

sampling seed 0


In [46]:
for model in ["ddpm"]:
    for idx in [0,2,3,5,6]: # 0,2,3,4,5
        plugin_cuda = Plugins().get(model, n_iter = 5, device=device,
                               batch_size = 4096)
        plugin_cuda.fit(dfs[idx])

        save_path = Path(f"results/{country_names[idx]}/{model}/model.pkl")
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plugin_loaded = load_from_file(save_path)

        # Pastikan state_dict CPU bisa diload ke module CUDA
        state = plugin_loaded.model.diffusion.denoise_fn.state_dict()

        plugin_cuda.model.diffusion.denoise_fn.load_state_dict(state)
        plugin_cuda.model.diffusion.denoise_fn.to(device)

        # Generate memakai diffusion CUDA
        plugin_cuda.device = device
        plugin_cuda.model.device = device
        plugin_cuda.model.diffusion.to(device)
        plugin_cuda.model.diffusion.eval()

        # Untuk categorical diffusion, beberapa tensor mungkin bukan registered buffer
        if hasattr(plugin_cuda.model.diffusion, "num_classes_expanded"):
            plugin_cuda.model.diffusion.num_classes_expanded = (
                plugin_cuda.model.diffusion.num_classes_expanded.to(device)
            )

        if hasattr(plugin_cuda.model.diffusion, "offsets"):
            plugin_cuda.model.diffusion.offsets = plugin_cuda.model.diffusion.offsets.to(device)
        
        print(f"sampling {model} on {country_names[idx]} dataset. using device {plugin.device}")
        
#         save_to_file(save_path, plugin)
#         print("Model Saved to:", save_path)
        
        for i in range(5):
            # generate again without fit ulang
            print(f'sampling seed {i}')
#             data_syn = generate_tabddpm(plugin_cuda, count=len(dfs[idx]), sample_batch = min(len(dfs[idx]), 40000))
            data_syn = generate2(plugin_cuda, len(dfs[idx])).dataframe()
            data_syn.to_csv(f"results/{country_names[idx]}/{model}/gen_data_{i}.csv", index=False)

[2026-05-20T11:35:34.870803+0100][734438][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T11:35:34.872260+0100][734438][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T11:35:34.873454+0100][734438][CRITICAL] module plugin_great load failed
[2026-05-20T11:35:34.874879+0100][734438][CRITICAL] module disabled: /mnt/hum01-home01/p88346bn/.conda/envs/synthcity/lib/python3.9/site-packages/synthcity/plugins/generic/plugin_goggle.py
Epoch: 100%|██████████| 5/5 [00:07<00:00,  1.60s/it, loss=1.61]


sampling ddpm on Indonesia dataset. using device cuda
sampling seed 0
sampling seed 1
sampling seed 2
sampling seed 3
sampling seed 4


In [47]:
for model in ["ddpm"]:
    for idx in [0,2,3,5]:
        plugin_cuda = Plugins().get(model, n_iter = 5, device=device,
                               batch_size = 4096)
        plugin_cuda.fit(dfs[idx])

        save_path = Path(f"results/{country_names[idx]}/{model}/model.pkl")
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plugin_loaded = load_from_file(save_path)

        # Pastikan state_dict CPU bisa diload ke module CUDA
        state = plugin_loaded.model.diffusion.denoise_fn.state_dict()

        plugin_cuda.model.diffusion.denoise_fn.load_state_dict(state)
        plugin_cuda.model.diffusion.denoise_fn.to(device)

        # Generate memakai diffusion CUDA
        plugin_cuda.device = device
        plugin_cuda.model.device = device
        plugin_cuda.model.diffusion.to(device)
        plugin_cuda.model.diffusion.eval()

        # Untuk categorical diffusion, beberapa tensor mungkin bukan registered buffer
        if hasattr(plugin_cuda.model.diffusion, "num_classes_expanded"):
            plugin_cuda.model.diffusion.num_classes_expanded = (
                plugin_cuda.model.diffusion.num_classes_expanded.to(device)
            )

        if hasattr(plugin_cuda.model.diffusion, "offsets"):
            plugin_cuda.model.diffusion.offsets = plugin_cuda.model.diffusion.offsets.to(device)
        
        print(f"sampling {model} on {country_names[idx]} dataset. using device {plugin.device}")
        
#         save_to_file(save_path, plugin)
#         print("Model Saved to:", save_path)
        
        for i in range(5):
            # generate again without fit ulang
            print(f'sampling seed {i}')
#             data_syn = generate_tabddpm(plugin_cuda, count=len(dfs[idx]), sample_batch = min(len(dfs[idx]), 40000))
            data_syn = generate2(plugin_cuda, len(dfs[idx])).dataframe()
            data_syn.to_csv(f"results/{country_names[idx]}/{model}/gen_data_{i}.csv", index=False)

[2026-05-20T11:46:23.415551+0100][734438][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T11:46:23.417075+0100][734438][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T11:46:23.418228+0100][734438][CRITICAL] module plugin_great load failed
[2026-05-20T11:46:23.419590+0100][734438][CRITICAL] module disabled: /mnt/hum01-home01/p88346bn/.conda/envs/synthcity/lib/python3.9/site-packages/synthcity/plugins/generic/plugin_goggle.py
Epoch: 100%|██████████| 5/5 [00:01<00:00,  3.55it/s, loss=2.37]


sampling ddpm on Canada dataset. using device cuda
sampling seed 0
sampling seed 1
sampling seed 2
sampling seed 3
sampling seed 4


[2026-05-20T11:47:17.562656+0100][734438][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T11:47:17.565243+0100][734438][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T11:47:17.566381+0100][734438][CRITICAL] module plugin_great load failed
[2026-05-20T11:47:17.567730+0100][734438][CRITICAL] module disabled: /mnt/hum01-home01/p88346bn/.conda/envs/synthcity/lib/python3.9/site-packages/synthcity/plugins/generic/plugin_goggle.py
Epoch: 100%|██████████| 5/5 [00:04<00:00,  1.10it/s, loss=2.05]


sampling ddpm on UK dataset. using device cuda
sampling seed 0
sampling seed 1
sampling seed 2
sampling seed 3
sampling seed 4


[2026-05-20T12:59:34.876365+0100][734438][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T12:59:34.878914+0100][734438][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T12:59:34.880004+0100][734438][CRITICAL] module plugin_great load failed
[2026-05-20T12:59:34.881263+0100][734438][CRITICAL] module disabled: /mnt/hum01-home01/p88346bn/.conda/envs/synthcity/lib/python3.9/site-packages/synthcity/plugins/generic/plugin_goggle.py
Epoch: 100%|██████████| 5/5 [00:01<00:00,  3.32it/s, loss=2.43]


sampling ddpm on Rwanda dataset. using device cuda
sampling seed 0
sampling seed 1
sampling seed 2
sampling seed 3
sampling seed 4


[2026-05-20T13:00:23.994820+0100][734438][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T13:00:23.996987+0100][734438][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T13:00:23.998115+0100][734438][CRITICAL] module plugin_great load failed
[2026-05-20T13:00:23.999388+0100][734438][CRITICAL] module disabled: /mnt/hum01-home01/p88346bn/.conda/envs/synthcity/lib/python3.9/site-packages/synthcity/plugins/generic/plugin_goggle.py
Epoch: 100%|██████████| 5/5 [00:01<00:00,  2.50it/s, loss=2.26]


sampling ddpm on Adult dataset. using device cuda
sampling seed 0
sampling seed 1
sampling seed 2
sampling seed 3
sampling seed 4


In [41]:
data_syn.dataframe()

,AGE,HRSWK,INCTOT,WKSWORK,ABIDENT,SEX,TENURE,URBAN,WKFULL,BPLMOM,...,MARST,TRANWORK,RELATE,DEGREE,OCCISCO,YRIMM,MINORITY,RELIG,IND,BPL
0,NaN,0.0,526900.000000,52.000000,2,1,1,1,1,3,...,1,NaN,3,NaN,NaN,NaN,NaN,1,1,1
1,29,40.0,12811.843750,35.000538,2,2,1,2,1,3,...,2,3,1,8,2,NaN,13,16,12,7
2,47,0.0,38475.137379,44.531971,2,1,1,2,2,1,...,2,1,2,5,3,NaN,13,3,17,7
3,67,35.0,26108.269908,52.000000,2,2,1,1,1,1,...,2,1,2,4,4,NaN,13,16,NaN,7
4,43,40.0,77700.707725,52.000000,2,1,1,1,1,1,...,2,2,2,7,3,NaN,13,4,15,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177424,7,0.0,12811.843750,20.000000,2,2,1,2,2,1,...,6,NaN,1,1,3,NaN,13,2,19,7
177425,NaN,0.0,23219.460900,0.000000,1,1,1,1,NaN,1,...,3,1,1,2,7,NaN,13,16,7,7
177426,26,0.0,200.000000,21.501772,2,1,1,2,NaN,1,...,3,NaN,1,1,NaN,NaN,13,16,NaN,7
177427,84,0.0,29145.817965,40.000000,2,1,1,1,NaN,3,...,3,NaN,2,1,NaN,NaN,13,5,NaN,6


In [33]:
for model in ["nflow"]:
    for idx in range(1,2):
        print(f"Running {model} on {country_names[idx]} dataset.")
        plugin = Plugins().get(model, n_iter = 120, device=device,
                               batch_size = 500)
        plugin.fit(dfs[idx])

        save_path = Path(f"results/{country_names[idx]}/{model}/model.pkl")
        save_path.parent.mkdir(parents=True, exist_ok=True)
        
        save_to_file(save_path, plugin)
        print("Model Saved to:", save_path)
        
        for i in range(5):
            # generate again without fit ulang
            data_syn = plugin.generate(count=len(dfs[idx])).dataframe()
            data_syn.to_csv(f"results/{country_names[idx]}/{model}/gen_data_{i}.csv", index=False)

[2026-05-20T08:19:19.093737+0100][724736][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T08:19:19.095052+0100][724736][CRITICAL] load failed: module 'synthcity.plugins.generic.plugin_great' has no attribute 'plugin'
[2026-05-20T08:19:19.096209+0100][724736][CRITICAL] module plugin_great load failed
[2026-05-20T08:19:19.097677+0100][724736][CRITICAL] module disabled: /mnt/hum01-home01/p88346bn/.conda/envs/synthcity/lib/python3.9/site-packages/synthcity/plugins/generic/plugin_goggle.py


Running nflow on Fiji dataset.


100%|██████████| 120/120 [01:45<00:00,  1.14it/s]


Model Saved to: results/Fiji/nflow/model.pkl


In [ ]:
# save fitted plugin
# save fitted plugin
save_path = Path("babi/run/tvae_niter10.pkl")
save_path.parent.mkdir(parents=True, exist_ok=True)

save_to_file(save_path, plugin)
print("Saved to:", save_path)

In [ ]:
plugin_loaded = load_from_file(save_path)

# generate again without fit ulang
syn_loader = plugin_loaded.generate(count=1000)
syn_df = syn_loader.dataframe()

In [ ]:
plugin.save()

In [ ]:
plugin2 = Plugins().get("tvae", n_iter = 100, device='cuda')
plugin.load('tvae_canada.sct')

In [ ]:
dt1 = plugin.generate(len(dfs[0]))
dt2 = plugin.generate(len(dfs[0]))

### Evaluation

In [ ]:

# for i in os.listdir('/mnt/hum01-home01/p88346bn/test/project/bayes-ctgan-fix/ctgan-bbb/code/'):
#     print(i)
#     #     if '.py' in i: exec(open('code/'+i).read())

In [ ]:
# exec(open('/mnt/hum01-home01/p88346bn/test/project/bayes-ctgan-fix/ctgan-bbb/code/eval.py').read())

def to_pandas_df(x):
    if isinstance(x, pd.DataFrame):
        return x.copy()

    if hasattr(x, "dataframe"):
        df = x.dataframe
        if callable(df):
            df = df()
        return df.copy()

    raise TypeError(f"Cannot convert {type(x)} to pandas DataFrame")

In [ ]:
real_df = to_pandas_df(dfs[id1])
syn_df = to_pandas_df(dt1)

roc_val = cal_mean_roc(country_names[id1], real_df.copy(), syn_df.copy())
cio_val = cal_mean_cio(country_names[id1], real_df.copy(), syn_df.copy())
tcap_val = cal_mean_tcap(country_names[id1], real_df.copy(), syn_df.copy())

In [ ]:
exec(open('eval.py').read())
id1 = 0
roc_val = cal_mean_roc(country_names[id1],dfs[id1],dt1)
cio_val = cal_mean_cio(country_names[id1],dfs[id1],dt1)
tcap_val = cal_mean_tcap(country_names[id1],dfs[id1],dt1)

In [ ]:
pd.__version__

In [ ]:
print(roc_val)

In [ ]:
warnings.filterwarnings("ignore")
res = []
for i in range(len(dt_best)):
    id1 = dt_best['id'][i]
    dsname = dt_best['dataset'][i]
    folder = 'data_sim_20240528_ctgan' if dt_best['kl_weight'][i] == 'non-bayes' else 'data-sim-results'
    fb = 'bbb' if dt_best['kl_weight'][i] == 'non-bayes' or 'bbb' in dt_best['bbb_function'][i] else dt_best['bbb_function'][i]
    mcs = dt_best['mc_sample_train'][i]
    loss = 'vanilla' if 'f' in dt_best['bbb_function'][i] or '-v' in dt_best['bbb_function'][i] else 'wasserstein'
    for k in range(5):
        dt1 = pd.read_csv(f'pbb-vanilla/results_20240518/{folder}/best_{dsname}_{fb}_{loss}_{str(mcs)}_{str(k)}.csv',dtype=str)
        if cont_columns[id1] is not None: dt1[cont_columns[id1]] = dt1[cont_columns[id1]].astype(float)
        roc_val = cal_mean_roc(country_names[id1],dfs[id1],dt1)
        cio_val = cal_mean_cio(country_names[id1],dfs[id1],dt1)
        tcap_val = cal_mean_tcap(country_names[id1],dfs[id1],dt1)
#             print([roc_val[0], roc_val[1], cio_val,(roc_val[0]+roc_val[1]+cio_val)/3,tcap_val])
        res.append([country_names[id1],fb,loss,mcs,k,roc_val[0], roc_val[1], cio_val,(roc_val[0]+roc_val[1]+cio_val)/3,tcap_val])